# Aceleración, horizontes y entrelazamiento: ¿por qué la gravedad cambia por debajo de ~10⁻¹⁰ m/s²?

**Proyecto:** *Vacuum Particle Journey*, línea A (mecanismo).

**De dónde venimos.** El notebook de la suma de distorsiones mostró que una red de puentes ER solo puede hacer de materia oscura si los puentes se concentran **donde la gravedad es débil**, por debajo de ~10⁻¹⁰ m/s². La pregunta es: ¿hay un mecanismo físico conocido por el que el entrelazamiento del vacío dependa de la aceleración?

**Sí lo hay, y es entrelazamiento puro:**
1. **El efecto Unruh es entrelazamiento.** Un observador acelerado no ve el vacío vacío: ve un baño térmico a temperatura $T = \hbar a/(2\pi c k_B)$. La razón es que el vacío está entrelazado a través del horizonte que ve ese observador; al no tener acceso al otro lado, su parte del vacío es térmica (teorema de Bisognano–Wichmann). Es el mismo estado termofield doble del notebook de geometría: dos regiones entrelazadas unidas, en el lenguaje ER = EPR, por un puente.
2. **En un universo con energía oscura hay otro horizonte**, el cósmico, y la temperatura que ve un observador acelerado es (Deser y Levin, 1997):
$$2\pi\,\frac{k_B T}{\hbar} = \frac{1}{c}\sqrt{a^2 + a_\Lambda^2},\qquad a_\Lambda = c\sqrt{\Lambda/3} = cH_0\sqrt{\Omega_\Lambda}$$
   Cuando $a \gg a_\Lambda$ manda la aceleración propia; cuando $a \lesssim a_\Lambda$ manda el horizonte cósmico, **compartido por todos**.
3. **Milgrom (1999)** señaló que $T(a) - T(0)$ depende de la aceleración igual que la inercia en MOND. **Ho, Minic y Ng (2010)**, siguiendo a Verlinde, obtuvieron así la ley $g_N = \sqrt{a^2 + a_0^2} - a_0$ y una escala crítica $a_c = a_0/2\pi = cH/2\pi$.

**Novedad, con honestidad:** estas ideas ya están publicadas. Lo que aporta tu hipótesis es la lectura geométrica: el entrelazamiento del vacío a través de los horizontes *es* la red de puentes ER que ve cada cuerpo, y por debajo de $a_\Lambda$ esa red la fija el cosmos y no la masa local.

**Lo que hace este notebook:**
1. Comprueba numéricamente que la temperatura de Unruh **sale del entrelazamiento** (cadena de fermiones).
2. Calcula las temperaturas y los horizontes, y los compara con las aceleraciones **reales** de las galaxias de SPARC.
3. Pone a prueba con los datos de SPARC la ley de Milgrom / Ho–Minic–Ng, con escalas **fijadas por la cosmología** (sin ajustar nada), frente a la RAR empírica.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False

COL = {"Datos": "#222222", "Solo bariones": "#8a8a85", "NFW ΛCDM (c–M)": "#6250d6",
       "RAR (g† = 1.2e-10)": "#e87ba4", "RAR local (g† global)": "#eb6834", "RAR no local (ER)": "#1baf7a", "Instrumental": "#2a78d6", "Densidad": "#eda100", "Tamaño": "#e34948"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6
C_KMS = 299792.458
H0 = 70.0; H0_KPC = H0 / 1000.0
KPC_M = 3.0857e19
TO_MS2 = 1e6 / KPC_M                    # (km/s)^2/kpc -> m/s^2
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
    print("plotly no disponible: las figuras 3D se harán con matplotlib (estáticas)")
from scipy import fft as sfft
N_GRID = 128
H0_PLANCK, H0_SH0ES, OMEGA_L = 67.4, 73.0, 0.685
C_MS = 2.99792458e8; HBAR = 1.054571817e-34; KB = 1.380649e-23; MPC_M = 3.0857e22
def cH(H0): return C_MS * H0 * 1e3 / MPC_M                          # m/s²
A_L = cH(H0_PLANCK) * np.sqrt(OMEGA_L)
print(f"cH0 = {cH(H0_PLANCK):.2e} m/s² (Planck) · a_Λ = c√(Λ/3) = {A_L:.2e} m/s² · cH0/2π = {cH(H0_PLANCK)/(2*np.pi):.2e} m/s²")

## 1. La temperatura de Unruh sale del entrelazamiento

Cadena de fermiones libres en su estado fundamental (el "vacío" de la cadena). Cortamos la cadena en dos y calculamos el **Hamiltoniano de entrelazamiento** $K_A$ de una mitad, definido por $\rho_A \propto e^{-K_A}$. Para fermiones libres se obtiene de la matriz de correlaciones $C$ como $K = \ln[(1-C)/C]$ (Peschel, 2003).

Bisognano y Wichmann predicen que $K_A$ es la energía de cada punto **pesada por su distancia al corte**: $K_A = \frac{2\pi}{v}\int d\, \mathcal{H}(d)\,\mathrm{d}d$. Eso es un sistema a temperatura local $T(d) = v/(2\pi d)$: exactamente la temperatura de Unruh de un observador con aceleración $a = v^2/d$ (con la velocidad del sonido $v$ en lugar de $c$). En esta cadena $v = 2$, así que los acoplamientos de $K_A$ deben crecer como $\pi\,d$.

In [ ]:
Nf = 400
Hf = -(np.eye(Nf, k=1) + np.eye(Nf, k=-1))
w_, V_ = np.linalg.eigh(Hf); Cf = V_[:, : Nf // 2] @ V_[:, : Nf // 2].T
Lh = Nf // 2
z, U = np.linalg.eigh(Cf[:Lh, :Lh]); z = np.clip(z, 1e-13, 1 - 1e-13)
Kmod = U @ np.diag(np.log((1 - z) / z)) @ U.T
dist = np.arange(1, 13)
hop = np.array([abs(Kmod[Lh - d, Lh - d - 1]) for d in dist])
slope = np.polyfit(dist[:5], hop[:5], 1)[0]
print("Acoplamiento del Hamiltoniano de entrelazamiento a distancia d del corte:")
print("  d:     ", dist[:8]); print("  |K|:   ", hop[:8].round(3)); print("  π·d:   ", (np.pi * dist[:8]).round(3))
print(f"Pendiente cerca del corte: {slope:.3f} (Bisognano–Wichmann predice π = {np.pi:.3f}) ✓")
print("(Más lejos la precisión numérica de 64 bits se satura: el estado es casi puro y ln[(1−C)/C] pierde resolución.)")
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(dist, hop, "o", color="#1baf7a", label="Hamiltoniano de entrelazamiento (cálculo)")
ax.plot(dist, np.pi * dist, "--", color="#9a9a95", label="Bisognano–Wichmann: π·d (temperatura de Unruh)")
ax.set_xlabel("distancia al corte d (sitios)"); ax.set_ylabel("|acoplamiento| = energía / temperatura local")
ax.set_title("El vacío entrelazado se ve caliente cerca del horizonte", loc="left"); ax.legend(fontsize=8.5); plt.show()

## 2. Temperaturas y horizontes, con las aceleraciones reales de las galaxias

Para cada punto de cada curva de rotación de SPARC calculamos la aceleración observada $g = V^2/r$. Comparamos:
- la **temperatura de Unruh** de esa aceleración, con la **temperatura del horizonte cósmico** (Gibbons–Hawking);
- la distancia al **horizonte de Rindler** de ese cuerpo, $c^2/a$, con el **radio del horizonte cósmico**, $c/H_\Lambda$.

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    o = np.argsort(g["r"][ok])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name in ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        os.makedirs("synthetic", exist_ok=True)
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        np.savetxt(path, np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r], header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: galaxias SINTÉTICAS con halo NFW. Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = [k for k in ["NGC3198", "NGC2403", "DDO154"] if k in GAL]
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in keep + list(RNG.choice(rest, 40 - len(keep), replace=False))}
N_TOT = sum(len(g["r"]) for g in GAL.values())
print(f"Galaxias: {len(GAL)} | puntos: {N_TOT}")

In [ ]:
TO_MS2 = 1e6 / 3.0857e19
gobs = np.concatenate([g["vobs"] ** 2 / g["r"] for g in GAL.values() if np.all(g["r"] > 0)]) * TO_MS2
gobs = gobs[gobs > 0]
T_U = lambda a: HBAR * a / (2 * np.pi * C_MS * KB)
T_GH = T_U(A_L)
print(f"Temperatura del horizonte cósmico: {T_GH:.1e} K")
for a in [1e-8, 1e-9, 1e-10, 1e-11]:
    print(f"  a = {a:.0e} m/s²: T_Unruh = {T_U(a):.1e} K · T_total(de Sitter) = {T_U(np.sqrt(a**2 + A_L**2)):.1e} K · "
          f"horizonte de Rindler c²/a = {C_MS**2/a/MPC_M/1e3:.1f} Gpc")
print(f"Radio del horizonte cósmico c/H_Λ = {C_MS / (A_L / C_MS) / MPC_M / 1e3:.1f} Gpc")
frac = np.mean(gobs < A_L)
print(f"\nPuntos de curvas de rotación con a < a_Λ: {frac:.0%} (en esos puntos, el horizonte de Rindler está MÁS LEJOS que el cósmico)")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
aa = np.geomspace(1e-12, 1e-8, 200)
a1.loglog(aa, T_U(aa), color="#9a9a95", ls="--", label="Unruh sin energía oscura")
a1.loglog(aa, T_U(np.sqrt(aa**2 + A_L**2)), color="#2a78d6", label="con horizonte cósmico (Deser–Levin)")
a1.loglog(aa, T_U(np.sqrt(aa**2 + A_L**2)) - T_GH, color="#1baf7a", label="T(a) − T(0) (Milgrom 1999)")
a1.axvline(A_L, color="#eb6834", lw=1, ls=":"); a1.text(A_L * 1.1, 2e-32, "a_Λ", color="#eb6834")
a1.set_xlabel("aceleración [m/s²]"); a1.set_ylabel("temperatura [K]"); a1.legend(fontsize=8)
a1.set_title("Por debajo de a_Λ manda el horizonte cósmico", loc="left")
a2.hist(np.log10(gobs), bins=40, color="#c3c2b7")
a2.axvline(np.log10(A_L), color="#eb6834", lw=1.5, label="a_Λ = c√(Λ/3)")
a2.axvline(np.log10(cH(H0_PLANCK) / (2 * np.pi)), color="#1baf7a", lw=1.5, ls="--", label="cH₀/2π")
a2.set_xlabel("log10 aceleración observada [m/s²]"); a2.set_ylabel("puntos de SPARC"); a2.legend(fontsize=8.5)
a2.set_title("Las galaxias viven justo en la transición", loc="left")
plt.tight_layout(); plt.show()

## 3. La prueba con datos: ¿basta la cosmología para fijar la escala?

Dos leyes que relacionan la aceleración observada $a$ con la newtoniana $g_N$ (la que producen solo estrellas y gas):

- **RAR (McGaugh, empírica):** $a = \dfrac{g_N}{1 - e^{-\sqrt{g_N/g_\dagger}}}$
- **Ley de horizontes (Milgrom 1999; Ho, Minic y Ng 2010):** $g_N = \sqrt{a^2 + s^2} - s \;\Rightarrow\; a = \sqrt{g_N^2 + 2 s\, g_N}$

Para cada ley se ajusta su escala ($g_\dagger$ o $s$) con las 149 galaxias, y se comparan además **valores fijados por la cosmología, sin ajustar nada**:
- $s = a_\Lambda = c\sqrt{\Lambda/3}$ (la propuesta literal de Milgrom);
- $s = a_\Lambda/2\pi$ y $s = cH_0/2\pi$ (la escala de Ho, Minic y Ng), con $H_0$ de Planck y de SH0ES.

Como en los notebooks anteriores: Υ por galaxia con prior, y 5 % de error sistemático.

In [ ]:
U_G = np.geomspace(0.1, 1.5, 40); PRI = (np.log10(U_G / 0.5) / 0.1) ** 2
SC = np.geomspace(1e-11, 3e-9, 90)
def law(gN, s, kind):
    gN = np.clip(gN, 1e-16, None)
    return gN / (-np.expm1(-np.sqrt(gN / s))) if kind == "rar" else np.sqrt(gN**2 + 2 * s * gN)
def table(kind, floor=0.05):
    names = list(GAL); T = np.zeros((len(names), len(SC))); Dd = np.zeros_like(T); npts = 0
    for n, k in enumerate(names):
        g = GAL[k]; r = g["r"]; err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2); npts += len(r)
        gN = (g["vgas2"][None, :] + U_G[:, None] * g["vstar2"][None, :]) / r * TO_MS2          # u, N
        a = law(gN[:, None, :], SC[None, :, None], kind)                                        # u, s, N
        v = np.sqrt(np.clip(a * r / TO_MS2, 0, None))
        c = (((g["vobs"] - v) / err) ** 2).sum(-1); t = c + PRI[:, None]; i = t.argmin(0)
        T[n] = t[i, np.arange(len(SC))]; Dd[n] = c[i, np.arange(len(SC))]
    return dict(tot=T, dat=Dd, npts=npts)
TB = {k: table(k) for k in ["rar", "horizontes"]}
def at(tab, s):
    j = np.clip(np.searchsorted(SC, s), 1, len(SC) - 1); w = (np.log(s) - np.log(SC[j - 1])) / (np.log(SC[j]) - np.log(SC[j - 1]))
    return tab["dat"][:, j - 1] * (1 - w) + tab["dat"][:, j] * w
NG = len(GAL); NP = TB["rar"]["npts"]
rows = []
for kind, label in [("rar", "RAR (escala ajustada)"), ("horizontes", "Ley de horizontes (escala ajustada)")]:
    S = TB[kind]["tot"].sum(0); j = int(np.argmin(S)); chi = TB[kind]["dat"][:, j].sum()
    bs = [SC[np.argmin(TB[kind]["tot"][RNG.integers(0, NG, NG)].sum(0))] for _ in range(500)]
    rows.append(dict(modelo=label, escala=f"{SC[j]:.2e}", IC68=f"{np.percentile(bs,16):.2e}–{np.percentile(bs,84):.2e}",
                     chi2=round(chi), BIC=round(chi + (NG + 1) * np.log(NP))))
    if kind == "horizontes": S_FIT_H = SC[j]
    else: G_FIT = SC[j]
for label, s in [("Horizontes, s = a_Λ (Milgrom literal)", A_L), ("Horizontes, s = a_Λ/2π", A_L / (2 * np.pi)),
                 ("Horizontes, s = cH₀/2π (Planck)", cH(H0_PLANCK) / (2 * np.pi)), ("Horizontes, s = cH₀/2π (SH0ES)", cH(H0_SH0ES) / (2 * np.pi))]:
    chi = at(TB["horizontes"], s).sum()
    rows.append(dict(modelo=label, escala=f"{s:.2e}", IC68="fija (0 parámetros)", chi2=round(chi), BIC=round(chi + NG * np.log(NP))))
chi = at(TB["rar"], 1.2e-10).sum()
rows.append(dict(modelo="RAR, g† = 1.2e-10 (McGaugh)", escala="1.20e-10", IC68="fija", chi2=round(chi), BIC=round(chi + NG * np.log(NP))))
RES = pd.DataFrame(rows); RES["ΔBIC"] = RES.BIC - RES.BIC.min()
print(RES.to_string(index=False))

In [ ]:
gg = np.geomspace(1e-12, 1e-8, 200)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(gg, gg, ls="--", color="#c3c2b7", lw=1, label="Newton (sin materia oscura)")
ax.loglog(gg, law(gg, G_FIT, "rar"), color="#e87ba4", label=f"RAR ajustada (g† = {G_FIT:.2e})")
ax.loglog(gg, law(gg, S_FIT_H, "horizontes"), color="#1baf7a", label=f"ley de horizontes ajustada (s = {S_FIT_H:.2e})")
ax.loglog(gg, law(gg, cH(H0_PLANCK) / (2 * np.pi), "horizontes"), color="#1baf7a", ls=":", label="ley de horizontes, s = cH₀/2π (sin ajustar)")
ax.loglog(gg, law(gg, A_L, "horizontes"), color="#eb6834", ls=":", label="ley de horizontes, s = a_Λ (sin ajustar)")
ax.set_xlabel("aceleración newtoniana g_N [m/s²]"); ax.set_ylabel("aceleración observada [m/s²]")
ax.set_title("La escala cosmológica, sin ajustar, frente a los datos", loc="left"); ax.legend(fontsize=8)
plt.show()
r_s = S_FIT_H / A_L; r_c = S_FIT_H / (cH(H0_PLANCK) / (2 * np.pi))
print(f"Escala que piden los datos para la ley de horizontes: s = {S_FIT_H:.2e} m/s² = {r_s:.3f}·a_Λ = {r_c:.2f}·(cH₀/2π)")
print(f"Equivale a a_Λ dividido por {1/r_s:.1f} (2π = {2*np.pi:.2f})")

## 4. Qué hemos aprendido

Lee la tabla con estas preguntas:
- **¿Se ajusta la ley de horizontes tan bien como la RAR?** Si su ΔBIC con escala ajustada es pequeño, la forma que sale de la termodinámica de horizontes describe las galaxias tan bien como la fórmula empírica.
- **¿Acierta la cosmología la escala sin ajustar nada?** Compara el ΔBIC de $s = a_\Lambda$ (la propuesta literal) con el de $s = cH_0/2\pi$, y mira cuánto vale el cociente impreso bajo la figura. Si los datos piden $a_\Lambda$ dividido por algo cercano a 2π, ese factor es a la vez el punto débil y la pista: hay que **derivarlo**, no ponerlo a mano.
- **¿Qué H₀?** La diferencia entre Planck y SH0ES cambia la escala ~8 %: el ajuste galáctico podría, en principio, opinar sobre la tensión de Hubble si la conexión fuera exacta.

**Conexión con tu hipótesis:** el entrelazamiento del vacío a través de los horizontes es real (sección 1), y la aceleración de las galaxias cae justo donde el horizonte cósmico empieza a dominar (sección 2). En el lenguaje ER = EPR, por debajo de $a_\Lambda$ el "puente" que más pesa para un cuerpo ya no es el de su propio horizonte de aceleración, sino el que lo conecta con el horizonte cósmico, compartido por toda la galaxia. Es la condición que pedía la suma de distorsiones: más efecto donde la gravedad es débil.

**El problema abierto, ahora bien definido:** derivar, desde el entrelazamiento a través de dos horizontes, (1) la forma de la ley y (2) el factor 2π. Si una derivación da $s = cH_0/2\pi$ sin ajustes y la ley ajusta las galaxias como la RAR, tendrías una explicación de la materia oscura aparente desde el entrelazamiento con **cero parámetros libres**.

**Referencias:** Unruh (1976) · Bisognano y Wichmann (1975) · Peschel (2003) J. Phys. A 36 L205 · Deser y Levin (1997) arXiv:gr-qc/9706018 · Jacobson (1997) arXiv:gr-qc/9709048 · Milgrom (1999) arXiv:astro-ph/9805346 · Ho, Minic y Ng (2010) arXiv:1005.3537 · Verlinde (2011) arXiv:1001.0785 · McGaugh, Lelli y Schombert (2016) arXiv:1609.05917.